# 01 — Data Loading and Quality Control

Load raw scRNA-seq data and perform initial quality control.

In [ ]:
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import QualityControl, Normalize
from src.utils import setup_logging, load_config

setup_logging()
config = load_config('config/config.yaml')

In [ ]:
# Load data — replace with your actual file path
adata = sc.read_h5ad('data/raw/sample_data.h5ad')
print(f'Loaded {adata.n_obs} cells x {adata.n_vars} genes')

In [ ]:
# Quality Control
qc = QualityControl(adata, config=config)
adata = qc.calculate_qc_metrics()
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'])
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
# Filter cells and genes
adata = qc.filter_cells(
    min_genes=config['quality_control']['min_genes'],
    max_genes=config['quality_control']['max_genes'],
    percent_mt=config['quality_control']['percent_mt'],
)
adata = qc.filter_genes(min_cells=config['quality_control']['min_cells'])
print(f'After QC: {adata.n_obs} cells x {adata.n_vars} genes')

In [ ]:
adata.write('data/processed/01_qced.h5ad')